In [0]:
from pyspark.sql import functions as F

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudfiles.schemaLocation", "/Volumes/retailnew/bronze/schema_logs/pos")
        .option("cloudFiles.inferColumnTypes", "true")
        .load("/Volumes/retailnew/bronze/raw_data/pos/")
        .withColumn("ingest_ts", F.current_timestamp())
)

(
    df.writeStream
      .format("delta")
      .option("checkpointLocation", "/Volumes/retailnew/bronze/checkpoints/pos")
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .table("retailnew.bronze.pos")
)